In [2]:
import ctypes
from ctypes import wintypes

class SYSTEM_POWER_STATUS(ctypes.Structure):
    _fields_ = [
        ("ACLineStatus", wintypes.BYTE),
        ("BatteryFlag", wintypes.BYTE),
        ("BatteryLifePercent", wintypes.BYTE),
        ("Reserved1", wintypes.BYTE),
        ("BatteryLifeTime", wintypes.DWORD),
        ("BatteryFullLifeTime", wintypes.DWORD),
    ]

status = SYSTEM_POWER_STATUS()
if ctypes.windll.kernel32.GetSystemPowerStatus(ctypes.byref(status)):
    print(f"ACLineStatus: {status.ACLineStatus}")
    print(f"BatteryLifePercent: {status.BatteryLifePercent}%")
    print(f"BatteryLifeTime: {status.BatteryLifeTime} seconds")


ACLineStatus: 1
BatteryLifePercent: 255%
BatteryLifeTime: 4294967295 seconds


In [3]:
import ctypes
import uuid
from ctypes import wintypes

# https://stackoverflow.com/questions/72847468/ctypes-how-to-parser-buffer-content

class GUID(ctypes.Structure):
    _fields_ = [
        ("Data1", ctypes.c_uint32),
        ("Data2", ctypes.c_uint16),
        ("Data3", ctypes.c_uint16),
        ("Data4", ctypes.c_uint8 * 8)
    ]

# class GUID(ctypes.Structure):
#     _fields_ = [
#         ("Data1", wintypes.ULONG),
#         ("Data2", wintypes.USHORT),
#         ("Data3", wintypes.USHORT),
#         ("Data4", wintypes.BYTE * 8)
#     ]

    # def __str__(self):
    #     return f"{{{self.Data1:08X}-{self.Data2:04X}-{self.Data3:04X}-{''.join(f'{x:02X}' for x in self.Data4)}}}"
    # def __repr__(self):
    #     return f"GUID('{self}')"

    def __str__(self):
        return (f"{{{self.Data1:08x}-{self.Data2:04x}-{self.Data3:04x}-"
                f"{bytes(self.Data4[:2]).hex()}-{bytes(self.Data4[2:]).hex()}}}")

    @staticmethod
    def to_string(guid_ptr) -> str:
        return str(guid_ptr.contents)

    def __init__(self, guid = None):
        if guid is not None:
            data = uuid.UUID(guid)
            self.Data1 = data.time_low
            self.Data2 = data.time_mid
            self.Data3 = data.time_hi_version
            self.Data4[0] = data.clock_seq_hi_variant
            self.Data4[1] = data.clock_seq_low
            self.Data4[2:] = data.node.to_bytes(6, "big")



In [4]:
# https://learn.microsoft.com/en-us/windows/win32/api/powersetting/nf-powersetting-powergetactivescheme
# Define necessary constants and types
PowerGetActiveScheme = ctypes.windll.powrprof.PowerGetActiveScheme
# [out] A pointer that receives a pointer to a GUID structure. Use the LocalFree function to free this memory.
PowerGetActiveScheme.argtypes = [wintypes.HANDLE, ctypes.POINTER(ctypes.POINTER(GUID))]
PowerGetActiveScheme.restype = wintypes.DWORD

In [5]:
# Initialize variables
active_scheme_guid_ptr = ctypes.POINTER(GUID)() # Create a pointer to a GUID

# Call the function
result = PowerGetActiveScheme(None, ctypes.byref(active_scheme_guid_ptr)) 

In [6]:
GUID.to_string(active_scheme_guid_ptr)

'{9897998c-92de-4669-853f-b7cd3ecb2790}'

In [7]:
def get_friendly_name(scheme_guid: GUID) -> str:
    PowerReadFriendlyName = ctypes.windll.powrprof.PowerReadFriendlyName
    PowerReadFriendlyName.argtypes = [ctypes.c_void_p, ctypes.POINTER(GUID), ctypes.POINTER(GUID), ctypes.POINTER(GUID), ctypes.POINTER(ctypes.c_wchar), ctypes.POINTER(ctypes.c_uint32)]
    PowerReadFriendlyName.restype = ctypes.c_uint32

    buffer_size = ctypes.c_uint32(0)
    PowerReadFriendlyName(None, ctypes.byref(scheme_guid), None, None, None, ctypes.byref(buffer_size))
    print(buffer_size.value)
    # buffer = (ctypes.c_ubyte * buffer_size.value)()
    buffer = ctypes.create_unicode_buffer(buffer_size.value)
    result = PowerReadFriendlyName(None, ctypes.byref(scheme_guid), None, None, buffer, ctypes.byref(buffer_size))

    if result == 0:  # ERROR_SUCCESS
        return buffer#[:buffer_size.value]#.decode('utf-16')
    else:
        raise ctypes.WinError(result)

friendly_name = get_friendly_name(active_scheme_guid_ptr.contents)
#print(f"Active Power Scheme Friendly Name: {friendly_name}")


40


In [8]:
type(friendly_name.value)

str

In [ ]:
import ctypes
from ctypes import wintypes

# Define the required structures and constants
class GUID(ctypes.Structure):
    _fields_ = [
        ("Data1", wintypes.DWORD),
        ("Data2", wintypes.WORD),
        ("Data3", wintypes.WORD),
        ("Data4", wintypes.BYTE * 8)
    ]

NO_SUBGROUP_GUID = GUID(0, 0, 0, (0, 0, 0, 0, 0, 0, 0, 0))

# Load the Power Management API
powrprof = ctypes.windll.powrprof

# Define the PowerReadFriendlyName function
PowerReadFriendlyName = powrprof.PowerReadFriendlyName
PowerReadFriendlyName.argtypes = [
    wintypes.HKEY,
    ctypes.POINTER(GUID),
    ctypes.POINTER(GUID),
    ctypes.POINTER(GUID),
    wintypes.LPWSTR,
    ctypes.POINTER(wintypes.DWORD)
]
PowerReadFriendlyName.restype = wintypes.DWORD

def get_friendly_name(scheme_guid=None, subgroup_guid=None, setting_guid=None):
    buffer_size = wintypes.DWORD(0)
    result = PowerReadFriendlyName(
        None,
        ctypes.byref(scheme_guid) if scheme_guid else None,
        ctypes.byref(subgroup_guid) if subgroup_guid else None,
        ctypes.byref(setting_guid) if setting_guid else None,
        None,
        ctypes.byref(buffer_size)
    )

    if result == 0:  # ERROR_SUCCESS
        buffer = ctypes.create_unicode_buffer(buffer_size.value)
        PowerReadFriendlyName(
            None,
            ctypes.byref(scheme_guid) if scheme_guid else None,
            ctypes.byref(subgroup_guid) if subgroup_guid else None,
            ctypes.byref(setting_guid) if setting_guid else None,
            buffer,
            ctypes.byref(buffer_size)
        )
        return buffer.value
    else:
        raise ctypes.WinError(result)

# Example usage
friendly_name = get_friendly_name(active_scheme_guid) # subgroup_guid=NO_SUBGROUP_GUID
print(friendly_name)